# 07 — Performance and profiling

What the deployed scoring system costs, measured rather than assumed:
- the local inference time of the pipeline;
- the end-to-end latency of the FastAPI service;
- single scoring against batch scoring;
- where the time actually goes, with cProfile;
- whether ONNX Runtime buys anything.

## Loading the benchmark results

Produced by scripts, so the numbers can be reproduced:

- `scripts/profile_inference.py`
- `scripts/benchmark_api.py`
- `scripts/benchmark_batching.py`
- `scripts/benchmark_onnx.py`

Results land in `reports/performance/`.

In [1]:
import json

with open("../reports/performance/inference_benchmark.json") as f:
    inference = json.load(f)

with open("../reports/performance/api_benchmark.json") as f:
    api = json.load(f)

with open("../reports/performance/batching_benchmark.json") as f:
    batching = json.load(f)

inference, api, batching

({'n_rows': 200,
  'n_loops': 20,
  'avg_ms': 11.654050000197458,
  'min_ms': 10.858900001039729,
  'max_ms': 18.788399998811656,
  'stats_file': 'reports\\performance\\cprofile_inference.prof',
  'top20_file': 'reports\\performance\\cprofile_inference_top20.txt'},
 {'n_requests': 50,
  'avg_ms': 16.321420000058424,
  'median_ms': 14.43699999981618,
  'min_ms': 12.50369999979739,
  'max_ms': 100.83730000042124,
  'status_codes': {'200': 50}},
 {'single_total_ms_for_50_calls': 146.37039999979606,
  'single_avg_ms_per_row': 2.9274079999959213,
  'batch_total_ms_for_50_rows': 4.449999998541898,
  'batch_avg_ms_per_row': 0.08899999997083796,
  'speedup_factor_per_row': 32.89222472983287})

## The pipeline, on its own

How long the full scikit-learn pipeline takes: preprocessing, imputation, transformation, and the LightGBM call. This is the floor — no HTTP, no serialisation, no database.

In [2]:
inference

{'n_rows': 200,
 'n_loops': 20,
 'avg_ms': 11.654050000197458,
 'min_ms': 10.858900001039729,
 'max_ms': 18.788399998811656,
 'stats_file': 'reports\\performance\\cprofile_inference.prof',
 'top20_file': 'reports\\performance\\cprofile_inference_top20.txt'}

## The API, end to end

The latency of a real request: HTTP transport, payload validation, JSON in and out, the model call, and building the response.

The gap between this and the previous number is what the service layer costs, and it is usually larger than people expect.

In [3]:
api

{'n_requests': 50,
 'avg_ms': 16.321420000058424,
 'median_ms': 14.43699999981618,
 'min_ms': 12.50369999979739,
 'max_ms': 100.83730000042124,
 'status_codes': {'200': 50}}

## Batch against single scoring

Scoring several applicants in one call amortises the costs that do not depend on the number of rows: Python call overhead, pandas and numpy conversions, scikit-learn's validation, and the preprocessing setup.

**This measurement is taken at the pipeline layer**, not through the API. The two are not the same number, and the second is the one a client would experience: it carries HTTP and per-row database logging that the local measurement does not. The end-to-end gain is necessarily smaller, and it is the one worth quoting.

In [4]:
batching

{'single_total_ms_for_50_calls': 146.37039999979606,
 'single_avg_ms_per_row': 2.9274079999959213,
 'batch_total_ms_for_50_rows': 4.449999998541898,
 'batch_avg_ms_per_row': 0.08899999997083796,
 'speedup_factor_per_row': 32.89222472983287}

## Where the time goes, with cProfile

The profile puts the cost in four places: scikit-learn's input validation, the imputation, the pandas/numpy conversions, and the preprocessing — with the LightGBM call itself a smaller share than any of them.

That is the useful finding here. The model is not the bottleneck; the wrapping around it is. Optimising the model would move almost nothing.

## ONNX Runtime, as an experiment

A best-effort ONNX conversion of the LightGBM classifier, with preprocessing left in Python.

What it showed: ONNX Runtime does speed up the classifier, the gain is real but modest, and it only touches the part of the pipeline that the profile says is not the bottleneck.

So ONNX stays a **proof of concept, not a retained optimisation**. Presenting it as a finished production path would misdescribe both what was built and what it bought.

In [7]:
from pathlib import Path
import json
import pandas as pd

# Find the project root automatically
ROOT = Path().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

perf_dir = ROOT / "reports" / "performance"

# Load the benchmarks
with open(perf_dir / "inference_benchmark.json", encoding="utf-8") as f:
    inference = json.load(f)

with open(perf_dir / "api_benchmark.json", encoding="utf-8") as f:
    api = json.load(f)

with open(perf_dir / "batching_benchmark.json", encoding="utf-8") as f:
    batching = json.load(f)

# ONNX (optional)
onnx_path = perf_dir / "onnx_benchmark.json"

if onnx_path.exists():
    with open(onnx_path, encoding="utf-8") as f:
        onnx = json.load(f)
else:
    onnx = {
        "status": "missing",
        "interpretation": "Benchmark ONNX non disponible."
    }

inference, api, batching, onnx

({'n_rows': 200,
  'n_loops': 20,
  'avg_ms': 11.654050000197458,
  'min_ms': 10.858900001039729,
  'max_ms': 18.788399998811656,
  'stats_file': 'reports\\performance\\cprofile_inference.prof',
  'top20_file': 'reports\\performance\\cprofile_inference_top20.txt'},
 {'n_requests': 50,
  'avg_ms': 16.321420000058424,
  'median_ms': 14.43699999981618,
  'min_ms': 12.50369999979739,
  'max_ms': 100.83730000042124,
  'status_codes': {'200': 50}},
 {'single_total_ms_for_50_calls': 146.37039999979606,
  'single_avg_ms_per_row': 2.9274079999959213,
  'batch_total_ms_for_50_rows': 4.449999998541898,
  'batch_avg_ms_per_row': 0.08899999997083796,
  'speedup_factor_per_row': 32.89222472983287},
 {'status': 'success',
  'pipeline_path': 'G:\\Mon Drive\\OC\\Projet_6\\credexp\\artifacts\\models\\pipeline.joblib',
  'holdout_path': 'G:\\Mon Drive\\OC\\Projet_6\\credexp\\data\\processed\\api_holdout.parquet',
  'onnx_model_path': 'G:\\Mon Drive\\OC\\Projet_6\\credexp\\artifacts\\models\\lightgbm_mode

In [8]:
summary = pd.DataFrame(
    [
        {
            "Expérience": "Pipeline scikit-learn local",
            "Métrique": "temps moyen / batch (ms)",
            "Valeur": inference.get("avg_ms"),
        },
        {
            "Expérience": "API FastAPI",
            "Métrique": "latence médiane (ms)",
            "Valeur": api.get("median_ms"),
        },
        {
            "Expérience": "Batching",
            "Métrique": "facteur d’accélération / ligne",
            "Valeur": batching.get("speedup_factor_per_row"),
        },
        {
            "Expérience": "ONNX Runtime",
            "Métrique": "facteur d’accélération",
            "Valeur": onnx.get("native_vs_onnx_speedup"),
        },
    ]
)

summary

,Expérience,Métrique,Valeur
0,Pipeline scikit-learn local,temps moyen / batch (ms),11.654050
1,API FastAPI,latence médiane (ms),14.437000
2,Batching,facteur d’accélération / ligne,32.892225
3,ONNX Runtime,facteur d’accélération,2.392050


In [9]:
status = onnx.get("status")

if status == "success":
    print("✅ Benchmark ONNX réussi")
    print(f"Temps pipeline natif total : {onnx.get('native_pipeline_total_ms'):.3f} ms")
    print(f"Temps ONNX total : {onnx.get('onnx_total_ms'):.3f} ms")
    print(f"Gain ONNX : x{onnx.get('native_vs_onnx_speedup'):.2f}")
elif status == "onnx_failed":
    print("⚠️ Conversion ou exécution ONNX échouée")
    print(onnx.get("onnx_error"))
    print(onnx.get("interpretation"))
else:
    print("Benchmark ONNX non disponible")

✅ Benchmark ONNX réussi
Temps pipeline natif total : 25.185 ms
Temps ONNX total : 10.529 ms
Gain ONNX : x2.39


## Conclusion

The system is fast enough for near-real-time use, and the measurements say why rather than asserting it:

- API latency is low;
- the local inference cost is under control;
- batching amortises the fixed costs, measured at the pipeline layer;
- ONNX adds a modest gain on the part that was not the bottleneck;
- the bottleneck is preprocessing and input validation, not the model.

What is kept, and why:
- FastAPI for serving;
- the scikit-learn pipeline, because having one object do preprocessing and inference is what keeps training and serving from diverging;
- `/predict` and `/predict_batch`, with the batch endpoint preferred whenever several applicants are scored together;
- latency watched through Prometheus and Grafana.